<a href="https://colab.research.google.com/github/stephleao/wmc-bc-data-analytics/blob/main/modulo-08/desafio/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio Probabilidade e Amostragem

Você é uma pesquisadora desenvolvendo uma análise sobre as características da força de trabalho nos estados brasileiros.

In [ ]:
# Importa as bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import norm

# Carregamento da base de dados via Google Drive
sheet_id = '1KnXUv8rB7VArbNL5c3ANZ1EB_s4daFxbM3hudxP4zqY'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

df = pd.read_csv(url)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Variaveis para reutilizacao

paleta = sns.color_palette("rocket") # paleta de cores
linha_cor = 'darkred'

n = len(df)               # Tamanho da amostra
z = 1.96                  # Z-core

faixa_tamanho = 1500          # Tamanho da faixa de renda
df_renda = df['renda']        # Espaco amostral de renda
renda_min = df_renda.min()    # Menor renda
renda_max = df_renda.max()    # Maior renda
renda_media = df_renda.mean() # Media da renda
renda_var = df_renda.var()    # Variancia da renda
renda_std = df_renda.std()    # Desvio padrao da renda

df_ingles = df['nivel_ingles']        # Espaco amostral de nivel de ingles
df_uf = df['uf']                      # Espaco amostral da uf/estado

df_escolaridade = df['escolaridade']  # Espaco amostral de escolaridade
ordem_escolaridade = ['Fundamental',
                      'Médio',
                      'Superior',
                      'Pós-graduação'] # da menor para a maior

## 1. Evento Complementar da Fluência em Inglês

Filtrar a coluna de inglês por nível "avançado" para achar a probabilidade de fluência $P(A)$. Calcular o complementar: $1 - P(A)$.

In [ ]:
ev_ingles_fluente = df_ingles.isin(['Avançado']) # Evento A
prob_c_ingles_fluente = 1 - ev_ingles_fluente.mean()

print(f'A probabilidade complementar de escolhermos alguém fluente é de {prob_c_ingles_fluente:.2%}.')

## 2. Probabilidade Condicional e Filtro Composto

Filtrar o df onde o Estado seja Alagoas ou Pará. Dentro desse subconjunto, calcular a proporção de pessoas com renda > 5000.

In [ ]:
df_al_pa = df[df_uf.isin(['AL', 'PA'])] # Espaço amostral condicionado ao evento B

renda_5k = 5000
# P(A|B), onde a renda e o evento A dentro de B
prob_renda_5k_dado_al_pa = (df_al_pa['renda'] > renda_5k).mean()

print(f'A probabilidade de escolhermos uma pessoa receber R$ {renda_5k} sendo de Alagoas ou do Pará é de {prob_renda_5k_dado_al_pa:.2%}.')

## 3. Distribuição Geométrica

1. Calcular a probabilidade base $p$ de ter ensino superior completo no Amazonas (Filtro por Estado e Escolaridade).
2. Aplicar a fórmula da Distribuição Geométrica para encontrar a chance de o sucesso ocorrer exatamente na 5ª tentativa: $P(X=5) = (1-p)^4 \cdot p$.

In [ ]:
ev_am = df_uf == 'AM' # Evento A
ev_esc_sup = df[ev_am]['escolaridade'] # Evento B
escolaridade_alvo = ordem_escolaridade[2]
prob_am_sup = ev_esc_sup.value_counts(normalize=True)[escolaridade_alvo] # P(A|B) - valor de p
k = 5 # Total de tentativas
prob_am_sup_5 = (1 - prob_am_sup) ** (k - 1) * prob_am_sup

print(f'A probabilidade da 5ª pessoa amazonense ter Ens. {escolaridade_alvo} é de {prob_am_sup_5:.2%}.')

## 4. Histograma e Densidade de Renda

Criar faixas de renda com intervalos de 1.500 reais (usar pd.cut()). Plotar o histograma de densidade (argumento stat='density' ou density=True) para visualizar e descrever a Função Densidade de Probabilidade (FDP) empírica da renda.

In [ ]:
renda_limites = np.arange(0, renda_max + faixa_tamanho, faixa_tamanho)

# Adiciona coluna categorica com as faixas
df['faixa_renda'] = pd.cut(df_renda, bins=renda_limites, right=False, include_lowest=True)
# Faz a contagem
contagem_renda = df['faixa_renda'].value_counts()

renda_moda_intervalo = contagem_renda.idxmax()
print(f'A renda da maioria ({contagem_renda.max()} pessoas) está na faixa de R$ {renda_moda_intervalo.left:.2f} - {renda_moda_intervalo.right:.2f}.\n')

In [ ]:
# Plota o grafico com a FDP
plt.figure(figsize=(8, 5))
sns.histplot(df_renda,
             bins=renda_limites,
             kde=True,
             color=paleta[0],
             stat='density')
plt.xticks(renda_limites)
plt.title('Função Densidade de Probabilidade da Renda')
plt.ylabel('Densidade (Probabilidade por R$)')
plt.xlabel('Renda (R$)')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 5. Média, Variância e Curva Normal

1. Calcular .mean() e .var() da coluna de renda.
2. Plotar a curva de distribuição normal teórica gerada a partir dessa média e desvio padrão sobreposta ao gráfico de barras.

In [ ]:
print(f'Média de renda da amostra: {renda_media:.2f}')
print(f'Variância da renda da amostra: {renda_var:.2f}')
print(f'Desvio padrão da renda da amostra: {renda_std:.2f}')

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_renda,
             bins=30,
             color=paleta[0],
             alpha=0.6,
             stat='density')

# Linha da Distribuicao Normal Teorica
x = np.linspace(renda_min - faixa_tamanho, renda_max + faixa_tamanho, 100)
y = norm.pdf(x, renda_media, renda_std)
plt.plot(x, y,
         color=linha_cor,
         linewidth=2,
         label=f'Normal teórica\n(μ={renda_media:.0f}, σ={renda_std:.0f})')
plt.title('Distribuição da Renda vs. Curva Normal')
plt.ylabel('Densidade')
plt.xlabel('Renda (R$)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 6. Distribuição Binomial com Grandes Amostras

1. Encontrar a probabilidade base de pós-graduação no conjunto de dados.
2. Calcular a probabilidade de encontrar 243 mil pessoas em 1 milhão utilizando a aproximação por Distribuição Binomial (ou aproximação Normal da Binomial devido ao tamanho de $n=1.000.000$).

## 7. Função de Densidade Acumulada Discreta - FDA

Obter a contagem de frequência relativa da coluna 'Escolaridade' organizada por ordem de nível de instrução. Aplicar o método `.cumsum()` para calcular e exibir a distribuição acumulada de forma discreta.

In [ ]:
# Calcula a probabilidade e coloca na ordem estabelecida
fda_escolaridade = df_escolaridade.value_counts(normalize=True).reindex(ordem_escolaridade).to_frame()
# Calcula a FDA
fda_escolaridade['FDA'] = fda_escolaridade.cumsum()
fda_escolaridade

## 8. Margem de Erro de Proporção

Encontrar a proporção amostral $\hat{p}$ de pessoas com inglês intermediário. Aplicar a fórmula de margem de erro para proporções populacionais usando o valor crítico $Z$ (Geralmente assume-se 95% de confiança, $Z=1.96$, caso o enunciado não dite outro).

In [ ]:
p_amostral = df_ingles.value_counts(normalize=True) # proporcao amostral

# Calcula a margem de erro
me = z * np.sqrt((p_amostral * (1 - p_amostral)) / n)

# Extrai o valor de  Intermediario
nivel_alvo = 'Intermediário'
p_ing_intmd = p_amostral.loc[nivel_alvo]
me_ing_intmd = me.loc[nivel_alvo]

print(f'Proporção amostral de pessoas com inglês intermediário: {p_ing_intmd:.2%}')
print(f'Margem de erro (95% de confiança): ± {me_ing_intmd:.2%}')

## 9. Probabilidade na Distribuição Normal

Utilizando a média encontrada na Questão 5, calcular o ponto correspondente a "Renda Média + 1.000". Usar a função de sobrevivência (sf ou $1 - CDF$) da distribuição normal da biblioteca scipy.stats para estimar a probabilidade de encontrar o grupo de 60 pessoas com essa característica.

## 10. Filtro Multivariável - Sudeste

Aplicar filtros simultâneos, onde: região = Sudeste, gênero = homem, escolaridade = Ensino Fundamental e renda > 2000. Calcular a proporção sobre o subgrupo.

In [ ]:
renda_2k = 2000
escolaridade_alvo = ordem_escolaridade[0]

ev_sudeste = df_uf.isin(['ES', 'MG', 'RJ', 'SP'])     # Evento A
ev_homem = df['genero'] == 'M'                        # Evento B
ev_fundamental = df_escolaridade == escolaridade_alvo # Evento C
ev_renda_2k = df_renda > renda_2k                     # Evento D

# P(A&B&C&D)
prob_sud_hom_fund_2k = ((ev_sudeste) & (ev_homem) & (ev_fundamental) & (ev_renda_2k)).mean()

print(f'A probabilidade de escolhermos um homem do Sudeste com Ens. {escolaridade_alvo} e renda de mais de R$ {renda_2k} é de {prob_sud_hom_fund_2k:.2%}.')